# Data

> Every input, downloaded once and cached.


In [ ]:
#| default_exp data

In [ ]:
#| export
import json
import zipfile

import geopandas as gpd
import osmnx as ox
import pandas as pd
import requests
import shapely
from shapely.geometry import LineString

from tunnel.config import BNG, COD_DIR, NPS_DIR, PRICE_YEAR, RAW, STUDY_BBOX, WGS84

In [ ]:
import tempfile
from pathlib import Path

from fastcore.test import test_eq

## Caching

Everything here follows one pattern: if the file isn't in `data/raw/` yet, make it, then read it. `cached` does the
first half. `make` is given the path to write.


In [ ]:
#| export
def cached(path, make):
    "`path`, after calling `make(path)` to create it if it doesn't exist yet."
    if not path.exists():
        make(path)
    return path

In [ ]:
calls = []
with tempfile.TemporaryDirectory() as d:
    p = Path(d) / "x.txt"
    for _ in range(3):
        cached(p, lambda p: (calls.append(1), p.write_text("hi")))
    test_eq(p.read_text(), "hi")
test_eq(len(calls), 1)

Most files are a plain download:


In [ ]:
#| export
HEADERS = {"User-Agent": "tunnel-stuff (Rotherhithe crossing research)"}


def download(url, **kwargs):
    "A function that saves `url` to the path it's given, for use with `cached`."

    def make(path):
        r = requests.get(url, headers=HEADERS, timeout=600, **kwargs)
        r.raise_for_status()
        path.write_bytes(r.content)

    return make

## OpenStreetMap

Street networks and map features come through osmnx, which writes its own files.


In [ ]:
#| export
def street_graph(kind):
    "The OSM `walk` or `bike` network for the study area."
    path = RAW / f"{kind}.graphml"
    make = lambda p: ox.save_graphml(ox.graph_from_bbox(STUDY_BBOX, network_type=kind, simplify=True), p)
    return ox.load_graphml(cached(path, make))


OSMNX_MIRRORS = ["https://overpass-api.de/api", "https://maps.mail.ru/osm/tools/overpass/api"]


def with_mirrors(fetch):
    "Call `fetch()` with osmnx pointed at each Overpass mirror in turn, until one works."
    for url in OSMNX_MIRRORS:
        ox.settings.overpass_url = url
        try:
            return fetch()
        except Exception:
            if url == OSMNX_MIRRORS[-1]:
                raise
        finally:
            ox.settings.overpass_url = OSMNX_MIRRORS[0]


def osm_features(name, tags, bbox=STUDY_BBOX):
    "OSM features matching `tags` in `bbox`, cached as `data/raw/<name>.gpkg`."

    def make(path):
        f = with_mirrors(lambda: ox.features_from_bbox(bbox, tags=tags)).reset_index()
        f[[c for c in ("element", "id", "name", "geometry") if c in f]].to_file(path)

    return gpd.read_file(cached(RAW / f"{name}.gpkg", make))

In [ ]:
#| export
def stations():
    "Stations (and piers) in the study area, as points."
    st = osm_features("stations", {"public_transport": "station"})[["name", "geometry"]]
    return st.assign(geometry=st.geometry.representative_point())


def thames():
    "The River Thames centre line, in BNG."
    rivers = osm_features("thames", {"waterway": "river"})
    return rivers[rivers.name == "River Thames"].to_crs(BNG)

In [ ]:
st = stations()
test_eq({"Canada Water", "Canary Wharf", "Rotherhithe"} <= set(st.name), True)
river = shapely.line_merge(thames().union_all())
river.geom_type, round(river.length / 1000, 1)

('LineString', 13.6)

A few things are easier to ask the Overpass API for directly. Its main server is often busy, so we fall back to a
mirror.


In [ ]:
#| export
OVERPASS = ["https://overpass-api.de/api/interpreter", "https://maps.mail.ru/osm/tools/overpass/api/interpreter"]


def overpass(query, name):
    "Elements returned by an Overpass `query`, cached as `data/raw/<name>.json`."

    def make(path):
        for url in OVERPASS:
            r = requests.post(url, data={"data": query}, headers=HEADERS, timeout=180)
            if r.ok:
                break
        r.raise_for_status()
        path.write_text(r.text)

    return json.loads(cached(RAW / f"{name}.json", make).read_text())["elements"]


def ways_to_lines(elements):
    "Overpass ways (fetched with `out geom`) as BNG lines."
    lines = [LineString([(p["lon"], p["lat"]) for p in e["geometry"]]) for e in elements]
    names = [e.get("tags", {}).get("name", "") for e in elements]
    return gpd.GeoDataFrame({"name": names}, geometry=lines, crs=WGS84).to_crs(BNG)


def jubilee_tunnels():
    "The Jubilee line's tunnels near the crossing, in BNG."
    q = """[out:json][timeout:60];
    way["railway"="subway"]["tunnel"="yes"]["name"~"Jubilee"](51.495,-0.050,51.510,-0.010);out tags geom;"""
    return ways_to_lines(overpass(q, "jubilee_tunnels_osm"))

In [ ]:
#| export
def buildings(bbox, name="buildings_crossing"):
    "Building footprints inside `bbox` (west, south, east, north in lon/lat), in BNG."
    w, s, e, n = bbox
    els = overpass(f'[out:json][timeout:90];way["building"]({s},{w},{n},{e});out geom;', name)
    rings = [[(p["lon"], p["lat"]) for p in el["geometry"]] for el in els if len(el.get("geometry", [])) > 3]
    return gpd.GeoDataFrame(geometry=[shapely.Polygon(r) for r in rings], crs=WGS84).to_crs(BNG)

In [ ]:
jubilee_tunnels().head(3)

,name,geometry
0,Jubilee Line,"LINESTRING (535649.237 179503.872, 535732.493 ..."
1,Jubilee Line,"LINESTRING (538439.753 179871.498, 538361.167 ..."
2,Jubilee Line,"LINESTRING (537181.56 180215.419, 537098.915 1..."


Some land can't be built on: parks, water, schools and places of worship. We sort these into three classes.


In [ ]:
#| export
CONSTRAINT_TAGS: dict[str, bool | str | list[str]] = {
    "leisure": ["park", "nature_reserve", "playground", "pitch", "garden", "common"],
    "landuse": ["recreation_ground", "cemetery", "grass", "village_green"],
    "amenity": ["school", "place_of_worship"],
    "natural": ["water"],
    "water": True,
}


def constraint_class(f):
    "`water`, `community` (schools, places of worship) or `open_space` for each OSM feature."
    cls = pd.Series("open_space", index=f.index)
    cls[f.get("amenity").isin(["school", "place_of_worship"])] = "community"
    cls[(f.get("natural") == "water") | f.get("water").notna()] = "water"
    return cls


def open_space(bbox):
    "Polygons of land that can't be built on in `bbox` (lon/lat), in BNG, with a `class` column."

    def make(path):
        f = ox.features_from_bbox(bbox, tags=CONSTRAINT_TAGS)
        f = f[f.geom_type.isin(["Polygon", "MultiPolygon"])]
        f.assign(**{"class": constraint_class(f)})[["class", "name", "geometry"]].reset_index(drop=True).to_file(path)

    return gpd.read_file(cached(RAW / "constraints_osm.gpkg", make)).to_crs(BNG)

In [ ]:
toy = pd.DataFrame({"amenity": ["school", None, None], "natural": [None, "water", None], "water": [None, None, None]})
test_eq(list(constraint_class(toy)), ["community", "water", "open_space"])

## ONS: boundaries and the Census

ONS boundaries come from its ArcGIS server, 2,000 features a page.


In [ ]:
#| export
ONS_ARCGIS = "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/{}/FeatureServer/0/query"
OA_CENTROIDS = "OA_December_2021_EW_PWC_V4"  # output areas, population-weighted centroids
OA_BOUNDARIES = "Output_Areas_2021_EW_BGC_V2"
MSOA_BOUNDARIES = "Middle_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V3"


def _arcgis_pages(service, bbox):
    params = {
        "where": "1=1", "outFields": "*", "f": "geojson", "outSR": 4326, "inSR": 4326, "geometry": ",".join(map(str, bbox)),
        "geometryType": "esriGeometryEnvelope", "spatialRel": "esriSpatialRelIntersects", "resultRecordCount": 2000,
    }  # fmt: skip
    offset = 0
    while True:
        r = requests.get(ONS_ARCGIS.format(service), params={**params, "resultOffset": offset}, timeout=120)
        r.raise_for_status()
        if not r.json().get("features"):
            return
        page = gpd.read_file(r.text)
        yield page
        offset += len(page)


def ons_layer(service, bbox=STUDY_BBOX):
    "ONS Open Geography features in `bbox`, in WGS84."
    make = lambda p: pd.concat(_arcgis_pages(service, bbox)).set_crs(WGS84).to_file(p)
    return gpd.read_file(cached(RAW / f"{service}.gpkg", make))

Households per output area come from Census table TS044.


In [ ]:
#| export
TS044_URL = "https://www.nomisweb.co.uk/output/census/2021/census2021-ts044.zip"


def households():
    "Census 2021 households by output area."
    with zipfile.ZipFile(cached(RAW / "ts044.zip", download(TS044_URL))) as z, z.open("census2021-ts044-oa.csv") as f:
        hh = pd.read_csv(f, usecols=["geography code", "Accommodation type: Total: All households"])
    return hh.set_axis(["OA21CD", "households"], axis=1)

In [ ]:
oa = ons_layer(OA_CENTROIDS).merge(households(), on="OA21CD")
print(f"{len(oa):,} output areas, {oa.households.sum():,} households")

1,439 output areas, 192,541 households


## House prices

Land Registry's Price Paid data, one file per postcode district. Postcodes become points via postcodes.io, which
takes 100 at a time. We keep every postcode we've looked up, so new ones are the only ones fetched.


In [ ]:
#| export
PPD_URL = "https://landregistry.data.gov.uk/app/ppd/ppd_data.csv"


def price_paid(district, since):
    "Every Price Paid record in a postcode district since `since` (YYYY-MM-DD)."
    params = {"header": "true", "limit": "all", "postcode": district, "min_date": since}
    return pd.read_csv(cached(RAW / f"ppd_{district}_{since}.csv", download(PPD_URL, params=params)), dtype=str)


def _lookup(postcodes):
    r = requests.post("https://api.postcodes.io/postcodes", json={"postcodes": postcodes}, timeout=60)
    r.raise_for_status()
    for item in r.json()["result"]:
        res = item["result"] or {}
        yield {"postcode": item["query"], "lon": res.get("longitude"), "lat": res.get("latitude")}


def geocode(postcodes):
    "Centroids (`postcode`, `lon`, `lat`) of `postcodes`, keeping every lookup in `data/raw/postcodes.csv`."
    path = RAW / "postcodes.csv"
    known = pd.read_csv(path) if path.exists() else pd.DataFrame(columns=["postcode", "lon", "lat"])
    todo = sorted(set(postcodes) - set(known.postcode))
    new = [row for i in range(0, len(todo), 100) for row in _lookup(todo[i : i + 100])]
    if new:
        known = pd.concat([known, pd.DataFrame(new)], ignore_index=True)
        known.to_csv(path, index=False)
    return known

In [ ]:
geocode(["SE16 5ST"]).query("postcode == 'SE16 5ST'")

,postcode,lon,lat
1664,SE16 5ST,-0.04419,51.50672


## Prices over time

To compare money from different years, we scale it by the ONS GDP deflator.


In [ ]:
#| export
DEFLATOR_URL = "https://www.ons.gov.uk/generator?format=csv&uri=/economy/grossdomesticproductgdp/timeseries/ybgb/qna"


def gdp_deflator():
    "ONS GDP deflator (YBGB), by year."
    d = pd.read_csv(cached(RAW / "gdp_deflator.csv", download(DEFLATOR_URL)), skiprows=8, names=["year", "index"])
    d = d[d.year.str.fullmatch(r"\d{4}")]
    return d.astype({"year": int, "index": float}).set_index("year")["index"]


def real(gbp, year):
    "`gbp` in `year` prices, converted to `PRICE_YEAR` prices."
    d = gdp_deflator()
    return gbp * d[PRICE_YEAR] / d[year]

In [ ]:
test_eq(real(100, PRICE_YEAR), 100)
f"£100m in 2017 is £{real(100, 2017):.0f}m in {PRICE_YEAR}"

'£100m in 2017 is £133m in 2025'

MHCLG publishes what residential land is worth in each local authority. The workbook isn't at a stable URL:
download "Land value estimates for policy appraisal 2023" from gov.uk to `data/raw/land_values_2023.xlsx`.


In [ ]:
#| export
def land_values():
    "MHCLG 2023 residential land values (£/ha) and density assumptions (homes/ha), by local authority."
    lv = pd.read_excel(RAW / "land_values_2023.xlsx", sheet_name="Residential", header=None, skiprows=5).iloc[:, 1:22]
    densities = [f"{d}_{p}" for d in ("low", "med", "high") for p in (25, 40, 50, 60, 75)]
    lv.columns = ["region", "la_code", "la", "dph_low", "dph_med", "dph_high"] + densities
    return lv.dropna(subset=["la"]).set_index("la")

In [ ]:
land_values().loc[["Southwark", "Tower Hamlets"], ["dph_high", "high_50"]]

,dph_high,high_50
la,,
Southwark,260.0,63640000
Tower Hamlets,400.0,98760000


## Land Registry: who owns what

Title plans come from the National Polygon Service. Owners come from CCOD and OCOD, which list every title held by a
company or public body, UK or overseas. A title in neither is held by private individuals. Tenure for every title is
in the NPS tenure file, coded `AF` (absolute freehold), `AL` (absolute leasehold), `PF` (possessory freehold) and so
on: the last letter is what matters.


In [ ]:
#| export
def title_plans(area):
    "NPS title plans that touch `area` (a BNG geometry)."
    files = sorted(NPS_DIR.glob("LR_POLY_FULL_*_[0-9].shp"))
    if not files:
        raise FileNotFoundError(f"No NPS shapefiles in {NPS_DIR}: set NPS_DIR in .env")
    parts = [gpd.read_file(f, bbox=area.bounds, engine="pyogrio") for f in files]
    g = pd.concat([p[p.intersects(area)] for p in parts], ignore_index=True).drop_duplicates("POLY_ID")
    g = gpd.GeoDataFrame(g.rename(columns=str.lower), geometry="geometry", crs=BNG)
    return g.assign(geometry=shapely.make_valid(g.geometry.values)).sort_values("poly_id").reset_index(drop=True)


COD_COLS = {
    "Title Number": "title_no", "Tenure": "tenure", "Property Address": "address", "Proprietor Name (1)": "owner",
    "Company Registration No. (1)": "company_no", "Proprietorship Category (1)": "owner_category",
}  # fmt: skip


def company_owners(titles):
    "CCOD/OCOD records for `titles`."
    files = sorted(COD_DIR.glob("[CO]COD_FULL_*.zip"))
    if not files:
        raise FileNotFoundError(f"No CCOD/OCOD zips in {COD_DIR}: set COD_DIR in .env")
    chunks = (
        c[c["Title Number"].isin(titles)]
        for f in files
        for c in pd.read_csv(f, usecols=list(COD_COLS), dtype=str, chunksize=500_000)
    )
    return pd.concat(chunks, ignore_index=True).rename(columns=COD_COLS).drop_duplicates("title_no")


def tenure_from_code(codes):
    "`Freehold` or `Leasehold` from NPS tenure codes like `AF` and `GL` (anything else is missing)."
    return codes.str[-1].map({"F": "Freehold", "L": "Leasehold"})


def tenures(titles):
    "Tenure of each of `titles`, from the NPS tenure file."
    path = next(NPS_DIR.glob("LR_TENURE_FULL_*.csv"))
    rows = pd.read_csv(path, header=None, usecols=[0, 2], names=["title_no", "code"], dtype=str, chunksize=2_000_000)
    t = pd.concat(c[c.title_no.isin(titles)] for c in rows).drop_duplicates("title_no").set_index("title_no")
    return tenure_from_code(t.code).rename("tenure")

In [ ]:
codes = pd.Series(["AF", "PF", "AL", "GL", "CN"])
test_eq(list(tenure_from_code(codes).fillna("?")), ["Freehold", "Freehold", "Leasehold", "Leasehold", "?"])